In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

_, base = expfolder( "system" );

In [ ]:
# Hyper parameters.
N = 128;  ρ = 1.0;  s = 0.1
N̂ = 128

# Extract critical noise.
folder = findfolder( N, ρ, s; base=base, crit=true )
f̂older = findfolder( N̂, ρ, s; base=base, crit=true )

# Unpack the kernel parameters.
γ = vec( readdlm( folder*"kernel-parameters-1.txt" ) )
println( "Kernel parameters: ", γ )

In [ ]:
# Import area and perimeter scaling values.
β = vec( readdlm( folder*"area-perimeter-scaling.txt" ) )[1:2]

# Import the cluster change frequency values.
θ = readdlm( folder*"optimal-rates-hat.txt" )[1,:]

# Extract the power law.
λ = readdlm( folder*"power-law.txt" )[1];

In [ ]:
# Import cluster mass distribution extracted from data.
F = round.( defInt, vec( readdlm( f̂older*"cluster-frequency.txt" ) ) )
f = F./sum( F )

# Unpack the probability map data.
ℓ = polyloglaw.( 1:N̂; β=β )
A = (ℓ.^2)./(4π)

# Construct the probability map.
Ω = [P( j; N=N̂, i=i, λℓ=θ[1], λr=θ[2], κ=θ[3] ) for i in 1:N̂, j in 1:N̂];

In [ ]:
# Simulate the master equations in the larger system size..
T = 1.0
NLL, f̂ = kernelloss( F, γ, (N̂, Ω, A, ℓ); T=T, verbose=true )

In [ ]:
# Plot bounds.
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( f[f .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(350,200), legend=:bottomleft )

# Population size.
plot!( plt, [N̂, N̂], 10.0.^[ymin,0]; color=:black, linestyle=:dot, lw=2, label="system size" )

# Plot mean-field and simulation cluster mass distributions.
scatter!( plt, f; color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, markersize=3, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)", legend=:bottomleft )
# plot!( plt; title=latexstring( "∣ λ - \\hat \\lambda ∣ = $(round( abs( λ - λ̂ ), digits=3 ))" ), titleposition=:left )

# saveplot( plt, "../../figures/extrapolated-cmd_N-$(N)"; dpi=600 )

In [ ]:
# Save the simulated distribution.
if false
    writedlm( f̂older*"cluster-distribution-hat.txt", f̂ )
end